## Retriever 
- 검색기 (Retriever) 는 저장된 벡터 데이터베이스에서 사용자의 질문과 관련된 문서를 검색하는 과정입니다.
- 사용자 질문에 가장 적합한 정보를 신속하게 찾아내는 것이 목적이며 RAG 시스템의 전반적인 성능과 직결되는 매우 중요한 과정입니다.

- **Retriever 기능**
    - 정확한 정보 제공
        - 사용자의 질문과 가장 관련성 높은 정보를 검색하여 시스템이 정확하고 유용한 답변을 생성할 수 있습니다.
    - 응답 시간 단축
        - 효율적인 검색 알고리즘을 사용하여 데이터베이스에서 적절한 정보를 빠르게 검색하여 전체적인 응답 시간을 단축 시킵니다.
    - 최적화
        - 효과적인 검색 과정을 통해 필요한 정보만을 추출함으로써 시스템 자원의 사용을 최적화하고 불필요한 데이터를 줄일 수 있습니다.

- **Retriever 동작 방식**
    1. 질문의 벡터화 
        - 사용자의 질문을 벡터 형태로 변환
        - 임베딩 단계와 유사한 기술을 사용하여 진행
        - 변환된 질문 벡터는 후속 검색 작업의 기준점으로 사용
    2. 벡터 유사성 비교 
        - 저장된 문서 벡터와 질문 벡터의 유사성을 계산 ( 코사인 유사성, MMR 방식 )

        - MMR 
            - 유사성 점수와 다양성 점수를 조합하여 최종 점수를 계산 
            query
                - 사용자로부터 입력받은 검색 쿼리입니다.
            k
                - 최종적으로 선택할 문서의 수입니다. 이 매개변수는 반환할 문서의 총 개수를 결정합니다.
            fetch_k
                - MMR 알고리즘을 수행할 때 고려할 상위 문서의 수입니다. 이는 초기 후보 문서 집합의 크기를 의미하며, 이 중에서 MMR에 의해 최종 문서가 k개 만큼 선택됩니다.
            lambda_mult
                - 쿼리와의 유사성과 선택된 문서 간의 다양성 사이의 균형을 조절합니다. 

    1. 상위 문서 선정 
        - 계산된 유사성 점수를 기준으로 상위 N개의 가장 관련성 높은 문서 선정
        - 상위 문서 선정을 통하여 사용자의 질문에 대한 답변 생성
        
    2. 검색기의 중요성 
        - RAG 시스템에서 **정보 검색의 질을 결정하는 핵심적인 역할**
        - 만약 검색기가 없을 경우 대규모 데이터베이스에서 관련 정보를 신속하고 정확하게 찾아내는 것이 매우 어려움
        
- **Sparse Retriever (스파스 검색기) & Dense Retriever (댄서 검색기)**
    - **Sparse Retriever**
        - Sparse Retriever는 TF-IDF나 BM25 와 같은 정보 검색 기법을 사용
            - **TF-IDF (Term Frequency-Inverse Document Frequency)**
                - 단어가 문서에 나타나는 빈도와 그 단어가 몇 개의 문서에서 나타나는지를 반영하여 단어의 중요도를 계산
                - 자주 나타나면서도 문서 집합 전체에서 드물게 나타나는 단어가 높은 가중치 할당
                
                → 단어의 빈도수를 기반으로 높은 가중치를 할당하는 방식
                
            
            - **BM25**
                - TF-IDF를 개선한 모델로, 문서의 길이를 고려하여 검색 정확도를 향상
                - 긴 문서와 짧은 문서 간의 가중치를 조정하여, 단어 빈도의 영향을 상대적으로 조절
                
                ```python
                [문서 1] 눈을 떠보니 커피 없는 세상에 와버렸다.
                [문서 2] 너도 커피 좋아해? 나도 커피 좋아해
                [문서 3] 저기 저 앞에 커피 파는 가게가 있다. 한 번 가볼래? 그래 내가 쏠게 마시러 가보자
                ```
                
                - 예를 들어 위와 같은 문서 3개가 있을 경우 BM25의 “커피”의 검색 결과는 어떻게 출력될까? ( 2 → 1 → 3 )
                    - BM25의 가중치 할당 조건
                        - 문서 내의 검색어 출현 빈도
                        - 다른 문서에는 검색어가 출현하지 않을 수록
                        - 문서 내용이 적을 수록
                        
            - Sparse Retriever의 특징은 각 단어의 존재 여부만을 고려하기 때문에 계산 비용이 낮고, 구현이 간단하다는 점입니다. 그러나 이 방법은 단어의 의미적 연관성을 고려하지 않으며, **검색 결과의 품질이 키워드의 선택에 크게 의존**합니다.
            
    - **Dense Retriever**
        - 딥러닝을 사용하여 문서와 쿼리를 연속적인 고차원 벡터로 인코딩
        - 문서의 의미적 내용을 보다 풍부하게 표현하여 키워드가 완벽하게 일치하지 않더라도 의미적으로 관련된 문서를 검색
        - 벡터 공간에서 거리(코사인 유사도)를 사용하여 쿼리와 가장 관련성 높은 문서를 검색
        - 언어의 문맥을 이해하는데 유리하며 복잡한 쿼리에 대해 더 정확한 검색 결과를 제공
        
    - **차이점 ( MBTI T와 F의 차이? )**
        - Sparse Retriever는 이산적인 키워드 기반의 표현, Dense Retriever는 연속적인 벡터 공간에서 의미적 표현 사용
        - Dense Retriever는 문맥과 의미를 더 깊이 파악할 수 있어, 키워드가 정확히 일치하지 않아도 관련 문서를 검색 가능
        - 복잡한 질문이나 자연어 쿼리에 대해서는 Dense Retriever가 더 적합할 수 있으며, 간단하고 명확한 키워드 검색에는 Sparse Retriever가 더 유용


## 주의 사항 
- langchain_upstage 은 Python 3.12 버전까지 지원중 

In [6]:
%%capture --no-stderr
%pip install langchain-opentutorial

In [2]:
# Install required packages
from langchain_opentutorial import package

package.install(
    [
        "langchain_opentutorial",
        "langchain_openai",
        "langchain_community",
        "langchain_text_splitters",
        "langchain_core",
        "langchain_upstage",
        "faiss-cpu"
    ],
    verbose=False,
    upgrade=False,
)


[notice] A new release of pip is available: 24.0 -> 25.1
[notice] To update, run: pip install --upgrade pip


In [3]:
# Configuration file to manage the API KEY as an environment variable
from dotenv import load_dotenv

# Load API KEY information
load_dotenv(override=True)

True

📌 **Creating a Vector Store (Using FAISS)**

Vector store에 구현된 유사도 검색(similarity search) 이나 MMR 과 같은 검색 메서드를 사용하여 vector store 내의 텍스트를 쿼리합니다.


In [4]:
from langchain_community.vectorstores import FAISS
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.document_loaders import TextLoader

#  TextLoader를 통한 파일 로드 
loader = TextLoader("./data/test.txt", encoding="utf-8")
documents = loader.load()
print(documents)
'''
[ 결과 값 ]
[
   Document("metadata="{
      "source":"./data/test.txt"
   },
   "page_content=""국가상징\n한 나라가 자신의 나라를 여러 나라에 알리기 위해 그 나라를 대표하는 내용을
     그림·문자·도형 등으로 나타낸 공식적인 상징이에요.\n세계의 각 나라마다 그 나라의 역사와 문화를 기초로 
     국기·국가·국화 등을 국가상징으로 정하여 국민들의 나라 사랑하는 마음을 하나로 모으고 세계적으로 
     나라 이미지를 알리기 위해 노력하고 있어요.\n우리나라의 국가상징으로는 태극기(국기), 애국가(국가), 
     무궁화(국화), 국새(나라도장), 나라문장이 있답니다.\n\n국새\n국새는 우리나라의 도장이에요.\n옛날에는 
     어보, 어새, 옥새, 국새 등 다양한 이름으로 불리어졌지만 현대에는 국새로 부른답니다.\n국새를 찍는다는 것은 
     나라에서 중요한 결정을 한다는 의미로 헌법 개정 공포문의 전문, 외교문서, 훈장증 등에 사용하고 있어요.
     \n현재 사용하고 있는 제5대 국새는 가로, 세로 10.4cm 정사각형으로 무게는 3.38kg이에요. 
     손잡이는 두 마리의 봉황이 앉아있는 모양이고, 봉황의 등 위로 활짝 핀 무궁화를 표현했어요.
     \n\n대한민국 나라문장\n문장은 외국에서 특정 가문이나 단체 또는 국가의 권위를 상징하는 
     장식적인 표시로 한 나라의 문장은 ‘국가 문장’ 또는 ‘국장’이라고 해요.\n우리나라의 문장은 
     태극문양을 무궁화 꽃잎 5장이 감싸고 ‘대한민국’ 글자가 새겨진 리본으로 그 테두리를 둘러싸고 있어요.
     \n1963년 12월 10일 ‘나라문장규정’을 제정하고, 외국기관에 발송되는 중요문서, 훈장 및 대통령 표창장, 
     재외공관의 건물 등에 대한민국의 상징으로 사용하고 있답니다.\n\n태극기\n1882년 박영효가 고종의 명을 받아 
     일본에 가면서 ‘태극·4괘 도안’의 기를 만들어 사용하였다는 기록이 있어요.\n고종은 1883년 3월 6일 왕명으로 
     이 ‘태극·4괘 도안’의 태극기를 국기로 제정·공포했지만 국기 만드는 방법을 구체적으로 정하지 않은 탓에 
     이후 다양한 형태의 국기가 사용되어 오다가 1948년 8월 15일 대한민국 정부가 수립되면서 태극기의 
     제작법을 통일할 필요성이 커짐에 따라, 정부는 1949년 10월 15일 「국기제작법고시」를 통해 국기 제작 방법을 
     확정·발표하였답니다.\n\n우리나라 이름 \n우리나라의 정식 이름은 “대한민국”이에요. 사용의 편의상 줄여서 
     부를 때에는 “대한” 또는 “한국”으로 쓸 수 있어요.\n영문으로는 “REPUBLIC OF KOREA”로 쓴답니다.
     \n\n애국가\n애국가(愛國歌)는 ‘나라를 사랑하는 노래’라는 뜻이에요. 우리나라는 애국가에 특별한 이름을 
     붙이지 않고 국가(國歌)로 사용하고 있어요.\n오늘날 불리고 있는 애국가 노랫말은 우리나라가 외세의 침략으로 
     위기에 처해있던 시기(1907년 전후)에 나라 사랑하는 마음과 우리 민족의 자주의식을 북돋우기 위해 만들어진 
     것으로 보여져요.\n그 후 여러 선각자의 손을 거쳐 현재와 같은 내용을 담게 되었는데 이 노랫말에 붙여진 곡조는 
     스코틀랜드 민요 ‘올드 랭 사인 (Auld Lang Syne)’ 이었답니다. 당시 해외에서 활동 중이던 작곡가 안익태(安益泰) 
     선생은 애국가에 남의 나라 곡을 붙여 부르는 것을 안타깝게 여겨 1935년에 오늘날의 애국가를 작곡하였다고 해요.
     \n1948년 대한민국 정부가 수립된 이후 현재의 애국가가 정부의 공식행사에서 불려지고, 교과서에도 실리면서 
     전국적으로 불려지기 시작했답니다.\n한 세기 가까운 세월 동안 슬플 때나 기쁠 때나 우리 겨레와 운명을 같이 해 
     온 애국가를 부를 때마다 우리 모두 선조들의 나라 사랑 정신을 새롭게 되새겨보아요.\n\n애국가 1절\n동해물과 
     백두산이 마르고 닳도록\n하느님이 보우하사 우리나라 만세\n무궁화 삼천리 화려 강산\n대한 사람 대한으로 길이 
     보전하세\n\n애국가 2절\n남산 위에 저 소나무 철갑을 두른 듯\n바람 서리 불변함은 우리 기상일세\n무궁화 
     삼천리 화려 강산\n대한 사람 대한으로 길이 보전하세\n\n애국가 3절\n가을 하늘 공활한데 높고 구름 없이\n밝은 
     달은 우리 가슴 일편단심일세\n무궁화 삼천리 화려 강산\n대한 사람 대한으로 길이 보전하세\n\n애국가 4절\n이 기
     상과 이 맘으로 충성을 다하여\n괴로우나 즐거우나 나라 사랑하세\n무궁화 삼천리 화려 강산\n대한 사람 대한
     으로 길이 보전하세\n"")"
]
'''

# 100 청크 기준으로 데이터 분할
text_splitter = CharacterTextSplitter(chunk_size=100, chunk_overlap=0)

split_docs = text_splitter.split_documents(documents) # Split into smaller chunks
#print(split_docs)
'''
[ 결과 값 ]
[
   Document("metadata="{
      "source":"./data/test.txt"
   },
   "page_content=""국가상징\n한 나라가 자신의 나라를 여러 나라에 알리기 위해 그 나라를 대표하는 내용을 그림·문자·도형 등으로 나타낸 공식적인 상징이에요.\n세계의 각 나라마다 그 나라의 역사와 문화를 기초로 국기·국가·국화 등을 국가상징으로 정하여 국민들의 나라 사랑하는 마음을 하나로 모으고 세계적으로 나라 이미지를 알리기 위해 노력하고 있어요.\n우리나라의 국가상징으로는 태극기(국기), 애국가(국가), 무궁화(국화), 국새(나라도장), 나라문장이 있답니다."")",
   "Document(metadata="{
      "source":"./data/test.txt"
   },
   "page_content=""국새\n국새는 우리나라의 도장이에요.\n옛날에는 어보, 어새, 옥새, 국새 등 다양한 이름으로 불리어졌지만 현대에는 국새로 부른답니다.\n국새를 찍는다는 것은 나라에서 중요한 결정을 한다는 의미로 헌법 개정 공포문의 전문, 외교문서, 훈장증 등에 사용하고 있어요.\n현재 사용하고 있는 제5대 국새는 가로, 세로 10.4cm 정사각형으로 무게는 3.38kg이에요. 손잡이는 두 마리의 봉황이 앉아있는 모양이고, 봉황의 등 위로 활짝 핀 무궁화를 표현했어요."")",
   "Document(metadata="{
      "source":"./data/test.txt"
   },
   "page_content=""대한민국 나라문장\n문장은 외국에서 특정 가문이나 단체 또는 국가의 권위를 상징하는 장식적인 표시로 한 나라의 문장은 ‘국가 문장’ 또는 ‘국장’이라고 해요.\n우리나라의 문장은 태극문양을 무궁화 꽃잎 5장이 감싸고 ‘대한민국’ 글자가 새겨진 리본으로 그 테두리를 둘러싸고 있어요.\n1963년 12월 10일 ‘나라문장규정’을 제정하고, 외국기관에 발송되는 중요문서, 훈장 및 대통령 표창장, 재외공관의 건물 등에 대한민국의 상징으로 사용하고 있답니다."")",
   "Document(metadata="{
      "source":"./data/test.txt"
   },
   "page_content=""태극기\n1882년 박영효가 고종의 명을 받아 일본에 가면서 ‘태극·4괘 도안’의 기를 만들어 사용하였다는 기록이 있어요.\n고종은 1883년 3월 6일 왕명으로 이 ‘태극·4괘 도안’의 태극기를 국기로 제정·공포했지만 국기 만드는 방법을 구체적으로 정하지 않은 탓에 이후 다양한 형태의 국기가 사용되어 오다가 1948년 8월 15일 대한민국 정부가 수립되면서 태극기의 제작법을 통일할 필요성이 커짐에 따라, 정부는 1949년 10월 15일 「국기제작법고시」를 통해 국기 제작 방법을 확정·발표하였답니다."")",
   "Document(metadata="{
      "source":"./data/test.txt"
   },
   "page_content=""우리나라 이름 \n우리나라의 정식 이름은 “대한민국”이에요. 사용의 편의상 줄여서 부를 때에는 “대한” 또는 “한국”으로 쓸 수 있어요.\n영문으로는 “REPUBLIC OF KOREA”로 쓴답니다."")",
   "Document(metadata="{
      "source":"./data/test.txt"
   },
   "page_content=""애국가\n애국가(愛國歌)는 ‘나라를 사랑하는 노래’라는 뜻이에요. 우리나라는 애국가에 특별한 이름을 붙이지 않고 국가(國歌)로 사용하고 있어요.\n오늘날 불리고 있는 애국가 노랫말은 우리나라가 외세의 침략으로 위기에 처해있던 시기(1907년 전후)에 나라 사랑하는 마음과 우리 민족의 자주의식을 북돋우기 위해 만들어진 것으로 보여져요.\n그 후 여러 선각자의 손을 거쳐 현재와 같은 내용을 담게 되었는데 이 노랫말에 붙여진 곡조는 스코틀랜드 민요 ‘올드 랭 사인 (Auld Lang Syne)’ 이었답니다. 당시 해외에서 활동 중이던 작곡가 안익태(安益泰) 선생은 애국가에 남의 나라 곡을 붙여 부르는 것을 안타깝게 여겨 1935년에 오늘날의 애국가를 작곡하였다고 해요.\n1948년 대한민국 정부가 수립된 이후 현재의 애국가가 정부의 공식행사에서 불려지고, 교과서에도 실리면서 전국적으로 불려지기 시작했답니다.\n한 세기 가까운 세월 동안 슬플 때나 기쁠 때나 우리 겨레와 운명을 같이 해 온 애국가를 부를 때마다 우리 모두 선조들의 나라 사랑 정신을 새롭게 되새겨보아요."")",
   "Document(metadata="{
      "source":"./data/test.txt"
   },
   "page_content=""애국가 1절\n동해물과 백두산이 마르고 닳도록\n하느님이 보우하사 우리나라 만세\n무궁화 삼천리 화려 강산\n대한 사람 대한으로 길이 보전하세"")",
   "Document(metadata="{
      "source":"./data/test.txt"
   },
   "page_content=""애국가 2절\n남산 위에 저 소나무 철갑을 두른 듯\n바람 서리 불변함은 우리 기상일세\n무궁화 삼천리 화려 강산\n대한 사람 대한으로 길이 보전하세"")",
   "Document(metadata="{
      "source":"./data/test.txt"
   },
   "page_content=""애국가 3절\n가을 하늘 공활한데 높고 구름 없이\n밝은 달은 우리 가슴 일편단심일세\n무궁화 삼천리 화려 강산\n대한 사람 대한으로 길이 보전하세"")",
   "Document(metadata="{
      "source":"./data/test.txt"
   },
   "page_content=""애국가 4절\n이 기상과 이 맘으로 충성을 다하여\n괴로우나 즐거우나 나라 사랑하세\n무궁화 삼천리 화려 강산\n대한 사람 대한으로 길이 보전하세"")"
]
'''


# 임베딩을 모델 정의
embeddings = OpenAIEmbeddings()

# 분할된 텍스트와 임베딩을 사용하여 FAISS 벡터 데이터베이스를 생성합니다.
db = FAISS.from_documents(split_docs, embeddings)

Created a chunk of size 247, which is longer than the specified 100
Created a chunk of size 251, which is longer than the specified 100
Created a chunk of size 250, which is longer than the specified 100
Created a chunk of size 280, which is longer than the specified 100
Created a chunk of size 108, which is longer than the specified 100
Created a chunk of size 540, which is longer than the specified 100


[Document(metadata={'source': './data/test.txt'}, page_content='국가상징\n한 나라가 자신의 나라를 여러 나라에 알리기 위해 그 나라를 대표하는 내용을 그림·문자·도형 등으로 나타낸 공식적인 상징이에요.\n세계의 각 나라마다 그 나라의 역사와 문화를 기초로 국기·국가·국화 등을 국가상징으로 정하여 국민들의 나라 사랑하는 마음을 하나로 모으고 세계적으로 나라 이미지를 알리기 위해 노력하고 있어요.\n우리나라의 국가상징으로는 태극기(국기), 애국가(국가), 무궁화(국화), 국새(나라도장), 나라문장이 있답니다.\n\n국새\n국새는 우리나라의 도장이에요.\n옛날에는 어보, 어새, 옥새, 국새 등 다양한 이름으로 불리어졌지만 현대에는 국새로 부른답니다.\n국새를 찍는다는 것은 나라에서 중요한 결정을 한다는 의미로 헌법 개정 공포문의 전문, 외교문서, 훈장증 등에 사용하고 있어요.\n현재 사용하고 있는 제5대 국새는 가로, 세로 10.4cm 정사각형으로 무게는 3.38kg이에요. 손잡이는 두 마리의 봉황이 앉아있는 모양이고, 봉황의 등 위로 활짝 핀 무궁화를 표현했어요.\n\n대한민국 나라문장\n문장은 외국에서 특정 가문이나 단체 또는 국가의 권위를 상징하는 장식적인 표시로 한 나라의 문장은 ‘국가 문장’ 또는 ‘국장’이라고 해요.\n우리나라의 문장은 태극문양을 무궁화 꽃잎 5장이 감싸고 ‘대한민국’ 글자가 새겨진 리본으로 그 테두리를 둘러싸고 있어요.\n1963년 12월 10일 ‘나라문장규정’을 제정하고, 외국기관에 발송되는 중요문서, 훈장 및 대통령 표창장, 재외공관의 건물 등에 대한민국의 상징으로 사용하고 있답니다.\n\n태극기\n1882년 박영효가 고종의 명을 받아 일본에 가면서 ‘태극·4괘 도안’의 기를 만들어 사용하였다는 기록이 있어요.\n고종은 1883년 3월 6일 왕명으로 이 ‘태극·4괘 도안’의 태극기를 국기로 제정·공포했지만 국기 만드는 방법을 구체적으로 정하지 않은 탓에 이후 다양한 형태의 국기가 사

📌 **1. VectorStore에서 VectorStoreRetriever 초기화(as_retriever)**

as_retriever 메서드는 VectorStore 객체를 기반으로 VectorStoreRetriever를 초기화하고 반환합니다. 
이 메서드를 통해 다양한 검색 옵션을 설정하여 사용자의 요구에 맞는 문서 검색을 수행할 수 있습니다.

매개변수(parameters)
    **kwargs: 검색 함수에 전달할 키워드 인자
    search_type: 검색 유형
        mmr
            - 관련성과 다양성을 동시에 검색하여 제공
            - 단순히 가장 관련성 높은 항목들만을 검색하는 대신, MMR은 쿼리에 대한 
              문서의 관련성 과 이미 선택된 문서들과의 차별성을 동시에 고려 
        similarity_score_threshold
            - 임계값을 적절히 설정함으로써 관련성이 낮은 문서를 필터링 하고, 질의와 가장 유사한 문서만 선별
            -  유사도 점수 임계값 검색
            - {"score_threshold": 0.8} 유사도 점수가 0.8 이상인 문서만 반환

    search_kwargs: 추가 검색 옵션
        k: 반환할 문서 수 (기본값: 4)
        score_threshold: similarity_score_threshold 검색의 최소 유사도 임계값
        fetch_k: MMR 알고리즘에 전달할 문서 수 (기본값: 20)
        lambda_mult: MMR 결과의 다양성 조절 (0-1 사이, 기본값: 0.5)
        filter: 문서 메타데이터 기반 필터링

반환값(return)
    VectorStoreRetriever: 초기화된 VectorStoreRetriever 객체

참고
    다양한 검색 전략 구현 가능 (유사도, MMR, 임계값 기반)
    MMR (Maximal Marginal Relevance) 알고리즘으로 검색 결과의 다양성 조절 가능
    메타데이터 필터링으로 특정 조건의 문서만 검색 가능
    tags 매개변수를 통해 검색기에 태그 추가 가능

주의사항
    search_type과 search_kwargs의 적절한 조합 필요
    MMR 사용 시 fetch_k와 k 값의 균형 조절 필요
    score_threshold 설정 시 너무 높은 값은 검색 결과가 없을 수 있음
    필터 사용 시 데이터셋의 메타데이터 구조 정확히 파악 필요
    lambda_mult 값이 0에 가까울수록 다양성이 높아지고, 1에 가까울수록 유사성이 높아짐


📌 **2. similarity_score_threshold 검색**

In [5]:
#유사도 점수 임계값 검색
# retriever = db.as_retriever(
#     #search_type="similarity_score_threshold" 
#     search_kwargs={
#         "k": 5,  # 반환 문서 
#         "score_threshold": 0.7  # 유사도 임계값
#     }
# )


# MMR( 검색 ) 
retriever = db.as_retriever(
    search_type="mmr", 
    search_kwargs={
        "k": 2, # 반환 문서 
        "fetch_k": 10, # MMR 알고리즘 전달 문서수 
        "lambda_mult": 0.1 # MMR 결과의 다양성 조절 (0~1, 기본값: 0.5, 0: 유사도 점수만 고려, 1: 다양성만 고려)
    }
)
# Perform the search
query = "애국가를 알려줘"
results = retriever.invoke(query) #invoke() 관련 문서 검색

#  Display the search results
for idx, doc in enumerate(results):
    print(f"\n🔍 [Search Result {idx + 1}]")
    print("📄 Document Content:", doc.page_content)
    print("🗂️ Metadata:", doc.metadata)
    print("=" * 60)


🔍 [Search Result 1]
📄 Document Content: 애국가 1절
동해물과 백두산이 마르고 닳도록
하느님이 보우하사 우리나라 만세
무궁화 삼천리 화려 강산
대한 사람 대한으로 길이 보전하세
🗂️ Metadata: {'source': './data/test.txt'}

🔍 [Search Result 2]
📄 Document Content: 태극기
1882년 박영효가 고종의 명을 받아 일본에 가면서 ‘태극·4괘 도안’의 기를 만들어 사용하였다는 기록이 있어요.
고종은 1883년 3월 6일 왕명으로 이 ‘태극·4괘 도안’의 태극기를 국기로 제정·공포했지만 국기 만드는 방법을 구체적으로 정하지 않은 탓에 이후 다양한 형태의 국기가 사용되어 오다가 1948년 8월 15일 대한민국 정부가 수립되면서 태극기의 제작법을 통일할 필요성이 커짐에 따라, 정부는 1949년 10월 15일 「국기제작법고시」를 통해 국기 제작 방법을 확정·발표하였답니다.
🗂️ Metadata: {'source': './data/test.txt'}


**Usage Example 3: Using** `config` **and** `**kwargs` **(Advanced Configuration)**

In [10]:
from langchain_core.runnables.config import RunnableConfig

# 실행 작업의 Tag, MetaData 지정 
# config는 실행 결과에는 영향을 미치지 않으나 정보 기록/ 로깅 목적으로 사용 
config = RunnableConfig(
    tags=["retrieval", "korea"], 
    metadata={"project": "korea_info"}  
)
docs = retriever.invoke(
    input="애국가를 알려줘", 
    config=config,  # Applying the config with tags and metadata
    search_kwargs={
        "k": 1,                   
        "score_threshold": 0.8   
    }
)

#  Display the search results
for idx, doc in enumerate(docs):
    print(f"\n🔍 [Search Result {idx + 1}]")
    print("📄 Document Content:", doc.page_content)
    print("🗂️ Metadata:", doc.metadata)
    print("=" * 60)

{'tags': ['retrieval', 'korea'], 'metadata': {'project': 'korea_info'}}

🔍 [Search Result 1]
📄 Document Content: 애국가 1절
동해물과 백두산이 마르고 닳도록
하느님이 보우하사 우리나라 만세
무궁화 삼천리 화려 강산
대한 사람 대한으로 길이 보전하세
🗂️ Metadata: {'source': './data/test.txt'}

🔍 [Search Result 2]
📄 Document Content: 태극기
1882년 박영효가 고종의 명을 받아 일본에 가면서 ‘태극·4괘 도안’의 기를 만들어 사용하였다는 기록이 있어요.
고종은 1883년 3월 6일 왕명으로 이 ‘태극·4괘 도안’의 태극기를 국기로 제정·공포했지만 국기 만드는 방법을 구체적으로 정하지 않은 탓에 이후 다양한 형태의 국기가 사용되어 오다가 1948년 8월 15일 대한민국 정부가 수립되면서 태극기의 제작법을 통일할 필요성이 커짐에 따라, 정부는 1949년 10월 15일 「국기제작법고시」를 통해 국기 제작 방법을 확정·발표하였답니다.
🗂️ Metadata: {'source': './data/test.txt'}


## 동적 구성(`ConfigurableField` 사용)

LangChain의 'ConfigurableField' 기능은 검색 구성의 **동적 조정**을 허용하여 쿼리 실행 중에 유연성을 제공합니다.

**주요 기능:**
- 런타임 검색 구성: 코어 검색기 설정을 수정하지 않고 검색 설정을 조정합니다.
- 향상된 추적성: 향상된 가독성과 디버깅을 위해 각 매개변수에 고유 식별자, 이름 및 설명을 할당합니다.
- `config`를 통한 유연한 제어: `config` 매개변수를 사전으로 사용하여 검색 구성을 동적으로 전달할 수 있습니다.


**사용 사례:**
- 검색 전략 전환: 검색 유형을 동적으로 조정합니다(예: `"similarity"`, `"mmr"`).
- 실시간 매개변수 조정: 쿼리 실행 중에 `k` , `score_threshold` 및 `fetch_k`와 같은 검색 매개변수를 수정합니다.
- 실험: 코드를 다시 작성하지 않고도 다양한 검색 전략과 매개변수 조합을 쉽게 테스트할 수 있습니다.


In [11]:
from langchain_core.runnables import ConfigurableField 

# Retriever Configuration Using ConfigurableField
retriever = db.as_retriever(search_kwargs={"k": 1}).configurable_fields(
    search_type=ConfigurableField(
        id="search_type", 
        name="Search Type",  # Name for the search strategy
        description="The search type to use",  # Description of the search strategy
    ),
    search_kwargs=ConfigurableField(
        id="search_kwargs",  
        name="Search Kwargs",  # Name for the search parameters
        description="The search kwargs to use",  # Description of the search parameters
    ),
)

The following examples demonstrate how to apply dynamic search settings using `ConfigurableField` in LangChain.


In [18]:
# ✅ Search Configuration 1: Basic Search (Top 3 Documents)

config_1 = {"configurable": {"search_kwargs": {"k": 3}}}

# Execute the query
docs = retriever.invoke("대한민국이 뭐야?", config=config_1)
print(retriever)

# default=VectorStoreRetriever("tags="[
#    "FAISS",
#    "OpenAIEmbeddings"
#    ],
#   vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x10f60c1a0>,
#   "search_kwargs="{
#       "k":1
#   }"
#) 
# fields="{
#    "search_type":"ConfigurableField(
#                                   id=""search_type",
#                                   "name=""Search Type",
#                                   "description=""The search type to use",
#                                   "annotation=None",
#                                   "is_shared=False
#                         )",
#    "search_kwargs":" ConfigurableField(
#                                   id=""search_kwargs",
#                                   "name=""Search Kwargs",
#                                   "description=""The search kwargs to use",
#                                   "annotation=None",
#                                   "is_shared=False
#                         )"
# }

# Display the search results
print("\n🔎 [Search Results - Basic Configuration (Top 3 Documents)]")
for idx, doc in enumerate(docs):
    print(f"📄 [Document {idx + 1}]")
    print(doc.page_content)
    print("=" * 60)

default=VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x10f60c1a0>, search_kwargs={'k': 1}) fields={'search_type': ConfigurableField(id='search_type', name='Search Type', description='The search type to use', annotation=None, is_shared=False), 'search_kwargs': ConfigurableField(id='search_kwargs', name='Search Kwargs', description='The search kwargs to use', annotation=None, is_shared=False)}

🔎 [Search Results - Basic Configuration (Top 3 Documents)]
📄 [Document 1]
우리나라 이름 
우리나라의 정식 이름은 “대한민국”이에요. 사용의 편의상 줄여서 부를 때에는 “대한” 또는 “한국”으로 쓸 수 있어요.
영문으로는 “REPUBLIC OF KOREA”로 쓴답니다.
📄 [Document 2]
대한민국 나라문장
문장은 외국에서 특정 가문이나 단체 또는 국가의 권위를 상징하는 장식적인 표시로 한 나라의 문장은 ‘국가 문장’ 또는 ‘국장’이라고 해요.
우리나라의 문장은 태극문양을 무궁화 꽃잎 5장이 감싸고 ‘대한민국’ 글자가 새겨진 리본으로 그 테두리를 둘러싸고 있어요.
1963년 12월 10일 ‘나라문장규정’을 제정하고, 외국기관에 발송되는 중요문서, 훈장 및 대통령 표창장, 재외공관의 건물 등에 대한민국의 상징으로 사용하고 있답니다.
📄 [Document 3]
애국가 1절
동해물과 백두산이 마르고 닳도록
하느님이 보우하사 우리나라 만세
무궁화 삼천리 화려 강산
대한 

In [31]:
# ✅ Search Configuration 2: Similarity Score Threshold (≥ 0.8)
# Score 0.8 이상 검색 

config_2 = {
    "configurable": {
        "search_type": "similarity_score_threshold",
        "search_kwargs": {
            "score_threshold": 0.8,  # Only return documents with a similarity score of 0.8 or higher
        },
    }
}

# Execute the query
docs = retriever.invoke("국새 알려줘", config=config_2)
print(retriever)
# Display the search results
print("\n🔎 [Search Results - Similarity Score Threshold ≥ 0.8]")
for idx, doc in enumerate(docs):
    print(f"📄 [Document {idx + 1}]")
    print(doc.page_content)
    print("=" * 60)

default=VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x10f60c1a0>, search_kwargs={'k': 1}) fields={'search_type': ConfigurableField(id='search_type', name='Search Type', description='The search type to use', annotation=None, is_shared=False), 'search_kwargs': ConfigurableField(id='search_kwargs', name='Search Kwargs', description='The search kwargs to use', annotation=None, is_shared=False)}

🔎 [Search Results - Similarity Score Threshold ≥ 0.8]
📄 [Document 1]
국새
국새는 우리나라의 도장이에요.
옛날에는 어보, 어새, 옥새, 국새 등 다양한 이름으로 불리어졌지만 현대에는 국새로 부른답니다.
국새를 찍는다는 것은 나라에서 중요한 결정을 한다는 의미로 헌법 개정 공포문의 전문, 외교문서, 훈장증 등에 사용하고 있어요.
현재 사용하고 있는 제5대 국새는 가로, 세로 10.4cm 정사각형으로 무게는 3.38kg이에요. 손잡이는 두 마리의 봉황이 앉아있는 모양이고, 봉황의 등 위로 활짝 핀 무궁화를 표현했어요.


In [30]:
# ✅ Search Configuration 3: MMR Search (Diversity and Relevance Balanced)
# MMR 검색

config_3 = {
    "configurable": {
        "search_type": "mmr",
        "search_kwargs": {
            "k": 2,            # Return the top 2 most diverse and relevant documents
            "fetch_k": 10,     # Initially fetch the top 10 documents before filtering for diversity
            "lambda_mult": 0.6 # Balance factor: 0.6 (0 = maximum diversity, 1 = maximum relevance)
        },
    }
}
# Execute the query using MMR search
docs = retriever.invoke("애국가 알려줘", config=config_3)

#  Display the search results
print("\n🔎 [Search Results - MMR (Diversity and Relevance Balanced)]")
for idx, doc in enumerate(docs):
    print(f"📄 [Document {idx + 1}]")
    print(doc.page_content)
    print("=" * 60)


🔎 [Search Results - MMR (Diversity and Relevance Balanced)]
📄 [Document 1]
애국가
애국가(愛國歌)는 ‘나라를 사랑하는 노래’라는 뜻이에요. 우리나라는 애국가에 특별한 이름을 붙이지 않고 국가(國歌)로 사용하고 있어요.
오늘날 불리고 있는 애국가 노랫말은 우리나라가 외세의 침략으로 위기에 처해있던 시기(1907년 전후)에 나라 사랑하는 마음과 우리 민족의 자주의식을 북돋우기 위해 만들어진 것으로 보여져요.
그 후 여러 선각자의 손을 거쳐 현재와 같은 내용을 담게 되었는데 이 노랫말에 붙여진 곡조는 스코틀랜드 민요 ‘올드 랭 사인 (Auld Lang Syne)’ 이었답니다. 당시 해외에서 활동 중이던 작곡가 안익태(安益泰) 선생은 애국가에 남의 나라 곡을 붙여 부르는 것을 안타깝게 여겨 1935년에 오늘날의 애국가를 작곡하였다고 해요.
1948년 대한민국 정부가 수립된 이후 현재의 애국가가 정부의 공식행사에서 불려지고, 교과서에도 실리면서 전국적으로 불려지기 시작했답니다.
한 세기 가까운 세월 동안 슬플 때나 기쁠 때나 우리 겨레와 운명을 같이 해 온 애국가를 부를 때마다 우리 모두 선조들의 나라 사랑 정신을 새롭게 되새겨보아요.
📄 [Document 2]
애국가 1절
동해물과 백두산이 마르고 닳도록
하느님이 보우하사 우리나라 만세
무궁화 삼천리 화려 강산
대한 사람 대한으로 길이 보전하세
